# 1. Import Libraries

In [10]:
!pip install ollama

import pandas as pd
import ollama


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


# 2. Read Data

In [11]:
df = pd.read_csv("../dataset/dataset.csv")

# 3. Create Prompt Functions

In [ ]:
def clean_news_article(text, model="llama3.1:8b"):
    system_prompt = """
You are a strict text cleaning tool.

### Core rules
- ONLY fix surface-level text issues (spacing, punctuation, broken encoding, HTML remnants)
- DO NOT guess missing words
- DO NOT rewrite, summarize, or change meaning
- DO NOT add or infer new information

### Text separation rule (IMPORTANT)
- If words are incorrectly merged due to formatting or encoding errors, you MAY separate them ONLY when the split is obvious and unambiguous.
- If the correct separation is unclear, DO NOT modify the word.

### Cleaning rules
You MAY remove the following:
- Email addresses (e.g. anything containing "@")
- Media / outlet footers (e.g. "CNBC INDONESIA RESEARCH", "REUTERS", "BBC", etc. when clearly standalone branding or signature lines)
- Author tags, contact lines, and publication signatures
- Standalone ellipses or noise lines (e.g. "...", "....")
- Navigation artifacts or UI remnants
- Repeated duplicate lines or sentences (keep only one instance)
- Isolated fragments that are clearly not part of the article body (e.g. single stray words, broken headers)

### Do NOT remove:
- Any meaningful sentence of the article body
- Any quoted speech or reported content, even if repetitive in style
- Numbers, dates, names, or facts unless clearly part of junk metadata

### Special rule
If a line is clearly a formatting artifact (not part of the article narrative), remove it entirely rather than “cleaning” it.

### Output rule
Return ONLY the cleaned text.
No explanations.
No commentary.
No formatting notes.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ]
    )

    return response["message"]["content"].strip()

def sentiment_analysis(text, model="llama3.1:8b"):
    system_prompt = """
You are a strict sentiment classifier for news articles.

Your task:
Classify the sentiment of the text as ONLY one of:
- Positive
- Neutral
- Negative

Rules:
- Output ONLY one word
- Do NOT explain
- Do NOT add punctuation
- Do NOT add extra text
- Do NOT guess beyond the text
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ]
    )

    return response["message"]["content"].strip()

# 4. Run

In [13]:
results = []
sentiments = []

for i, row in df.iterrows():
    cleaned = clean_news_article(row["konten"])
    sentiment = sentiment_analysis(cleaned)

    results.append(cleaned)
    sentiments.append(sentiment)

    print(i, cleaned, sentiment)

df["cleaned"] = results
df["ollama_label"] = sentiments

0 Jakarta, CNBC Indonesia - Amerika Serikat (AS) secara resmi memiliki Presiden baru yang disambut positif karena agenda pro-bisnis yang dibawanya. 
Presiden Amerika Serikat (AS) Donald Trump resmi dilantik pada Senin waktu AS (20/1/2025). 
Secara umum, dalam pidatonya, Trump mengungkapkan kebijakan perdagangan proteksionis yang sebelumnya dikhawatirkan akan dilakukan secara agresif tampaknya akan diterapkan lebih terukur. 
Trump membawa agenda ambisius meliputi reformasi perdagangan, imigrasi, pemangkasan pajak, dan deregulasi. Kebijakan ini berpotensi meningkatkan keuntungan korporasi AS, tetapi juga berisiko memicu inflasi yang dapat menekan saham, obligasi, dan mendorong bank sentral AS (The Fed) menaikkan suku bunga. 
Executive Order/Perintah Eksekutif 
Dilansir dari ABCnews, Trump pada malam pertama masa kepresidenan keduanya, mencabut 78 tindakan eksekutif, perintah eksekutif, dan memorandum presiden era Joe Biden sebagai bagian dari serangkaian tindakan eksekutif yang ia rencan

KeyboardInterrupt: 

# 5. Save to CSV

In [ ]:
df.to_csv("../outputs/dataset_ollama.csv")